# Ternary Interpolation Demo (Al-Si-Mg)

This notebook demonstrates the binary-to-ternary interpolation workflow for the Al-Si-Mg system. For use on the entire gliquid binary dataset, check out the [the online version](https://viz.whsunresearch.group/gliquid/ternary-interpolation/)

1. Define a helper for ternary interpolation/plotting.
2. Intialize binary fitted parameters.
3. Generate an interactive ternary plot.

In [ ]:
## UNCOMMENT THIS BLOCK FOR GOOGLE COLAB
# !git clone https://github.com/willwerj/gliquid_python.git
# %cd /content/gliquid_python
# !pip install -q .
# from pathlib import Path
# import gliquid.config as cfg
# cfg.set_data_dir(Path("/content/gliquid_python/data").resolve())

In [ ]:
import os
import gliquid.config as cfg

# By default cfg.project_root and cfg.data_dir are resolved from the installed package location.
# Override them here if your data lives elsewhere:
#   cfg.set_project_root("path/to/G_liquid")
#   cfg.set_data_dir("path/to/matrix_data")
print(f"Project root : {cfg.project_root}")
print(f"Data directory: {cfg.data_dir}")

# Materials Project API key — required for fetching DFT convex hull data.
# Replace with your own key from https://materialsproject.org/api
os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"

## 1 - Define the interpolation helper

This function formats fitted binary parameters to match the sorted ternary axis order, interpolates the ternary free-energy surface, and optionally writes an HTML output.

In [ ]:
import plotly.offline as ploff
from pathlib import Path
from gliquid.hsx_ternary import ternary_gtx_plotter

def plot_ternary_system(
    tern_sys,
    binary_data,
    param_format='comb-exp',
    temp_slider=(0, 300),
    t_incr=5,
    delta=0.025,
    save_html=False,
):
    """Interpolate ternary liquid free energy from binary parameters and return a Plotly figure."""
    if isinstance(tern_sys, str):
        tern_sys = tern_sys.split('-')
    tern_sys = sorted(tern_sys)

    # Keep only binaries that are subsets of the ternary system
    binary_dict = {
        sys_name: data.copy()
        for sys_name, data in binary_data.items()
        if set(sys_name.split('-')).issubset(set(tern_sys))
    }

    # Reorder systems to match adjacent ternary ordering; flip L1 sign if reversed
    items_to_remove = []
    items_to_add = {}
    for sys_name, data in binary_dict.items():
        components = sys_name.split('-')
        if abs(tern_sys.index(components[0]) - tern_sys.index(components[1])) != 1:
            items_to_remove.append(sys_name)
            data['params'][2] = -data['params'][2]
            new_sys_name = f"{components[1]}-{components[0]}"
            items_to_add[new_sys_name] = {'params': data['params']}

    for item in items_to_remove:
        binary_dict.pop(item)
    binary_dict.update(items_to_add)

    binary_L_dict = {sys_name: data['params'] for sys_name, data in binary_dict.items()}

    plotter = ternary_gtx_plotter(
        tern_sys,
        interp_type='linear',
        param_format=param_format,
        L_dict=binary_L_dict,
        temp_slider=list(temp_slider),
        T_incr=t_incr,
        delta=delta,
    )
    plotter.interpolate()
    plotter.process_data()
    tern_fig = plotter.plot_ternary()

    if save_html:
        out_file = Path(cfg.project_root) / 'figures' / f"{''.join(tern_sys)}_system.html"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        ploff.plot(tern_fig, filename=str(out_file), auto_open=False)
        print(f"Saved ternary figure: {out_file}")

    return tern_fig

## 2 - Define binary fitted parameters

Here are parameters for three of the binary systems in our database: Al-Si, Al-Mg and Mg-Si. All of these are fitted using the 'comb-exp' model, which is presently the default model we use for ternary interpolations

In [ ]:
fitted_data = {
    'Al-Si': {'params': [-13430, -1.516, -966.7, 0]},
    'Al-Mg': {'params': [804.7, 0.08031, 258.5, 0]},
    'Mg-Si': {'params': [-45190, 9.047, 1482, 0]},
}

for key, value in fitted_data.items():
    print(f"{key}: {value}")

## 3 - Interpolate and plot the ternary system

Run the interpolation and display the interactive ternary diagram.
Set `save_html=True` to also write an HTML file in the `figures` directory

In [ ]:
ternary_figure = plot_ternary_system(
    'Al-Si-Mg',
    fitted_data,
    param_format='comb-exp',
    t_incr=5, # Sets the temperature resolution - smaller values increase resolution but also increase computation time
    delta=0.01, # Sets the composition resolution - smaller values increase resolution but also increase computation time
    save_html=False,
)
ternary_figure.show()